<a href="https://colab.research.google.com/github/DivyaB723/Project_1/blob/main/FirstProject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pandas

In [ ]:
import requests
import pandas as pd
from datetime import datetime

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

all_records = []
start_year = datetime.now().year - 5
end_year = datetime.now().year

for year in range(start_year, end_year + 1):
   for month in range(1, 13):
    start_date = f"{year}-{month:02d}-01"
    if month == 12:
     end_date = f"{year+1}-01-01"
    else:
     end_date = f"{year}-{month+1:02d}-01"

    params = {
        "format" : "geojson",
        "starttime" : start_date,
        "endtime" : end_date,
        "minmagnitude" : 3

    }

    response = requests.get(url, params = params)
    if response.status_code != 200:
      print(f"Failed for {start_date} : {response.text[:200]}")
      continue

    try:
      data = response.json()
    except Exception as e:
      print(f"JSON error for{start_date} : {e}")
      continue

    for f in data["features"]:
      p = f["properties"]
      g = f["geometry"] ["coordinates"]
      all_records.append({
          "id" : f.get("id"),
          "time" : pd.to_datetime(p.get("time"), unit = "ms"),
          "updated" : pd.to_datetime(p.get("updated"), unit = "ms"),
          "latitude" : g[1] if g else None,
          "longitude" : g[0] if g else None,
          "depth_km" : g[2] if g else None,
          "mag" : p.get("mag"),
          "magType" : p.get("magType"),
          "place" : p.get("place"),
          "status" : p.get("status"),
          "tsunami" :p.get("tsunami"),
          "sig" : p.get("sig"),
          "net" : p.get("net"),
          "nst" : p.get("nst"),
          "dmin" : p.get("dmin"),
          "rms" : p.get("rms"),
          "gap" : p.get("gap"),
          "magError" : p.get("magError"),
          "depthError" : p.get("depthError"),
          "magNst" : p.get("magNst"),
          "locationSource" : p.get("locationSource"),
          "magSource" : p.get("magSource"),
          "types" : p.get("types"),
          "ids" : p.get("ids"),
          "sources" : p.get("sources"),
          "type" : p.get("type"),
          "felt" : p.get("felt"),
          "alert" : p.get("alert")

          })

df = pd.DataFrame(all_records)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df.head())
df.to_csv('output.csv', index=False)

df['country']= df['place'].str.split(',').str[-1].str.strip(',')
df.head()

df['alert'] = df['alert'].ffill()
df.head()

string_columns = ['place', 'id', 'types', 'sources', 'alert', 'status', 'locationSource', 'magType','net', 'type']
for col in string_columns:
   if col in df.columns:
      df[col] = (df[col].astype(str).str.replace(r'[@#?!$^]', '', regex=True).str.strip(','))
df.head()

numeric_columns = ['mag', 'depth_km', 'nst', 'dmin', 'rms', 'sig', 'gap', 'felt']
for col in numeric_columns:
   if col in df.columns:
      df[col] = pd.to_numeric(df[col], errors='coerce')
      if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())
df.head()

df.to_csv("earthquake_data.csv", index= False)



Rows: 106443
Columns: 28
           id                    time                 updated  latitude  \
0  us6000ddi8 2021-01-31 23:20:49.923 2021-04-16 19:02:44.040  -31.7493   
1  us6000dev6 2021-01-31 23:08:17.161 2021-04-16 19:03:47.040  -15.4902   
2  us6000dev5 2021-01-31 22:54:19.760 2021-04-16 19:03:47.040   19.7529   
3  us6000ddhs 2021-01-31 22:06:00.832 2021-04-16 19:02:43.040   28.1524   
4  us6000dev4 2021-01-31 21:51:14.016 2021-04-16 19:03:46.040   71.3212   

   longitude  depth_km  mag magType  \
0   -68.9337     17.27  4.7     mwr   
1  -177.2052    426.71  4.1      mb   
2   121.3159     46.73  4.7      mb   
3    57.2570     10.00  4.9      mb   
4    -3.7578     10.00  4.0      mb   

                                               place    status  ...  \
0        29 km SW of Villa Basilio Nievas, Argentina  reviewed  ...   
1                                        Fiji region  reviewed  ...   
2                    103 km SW of Basco, Philippines  reviewed  ...   
3    